# 01 · Bronze — dado cru, com rastreabilidade

```
landing (arquivos crus) ──► BRONZE (Iceberg) ──► Silver ──► Gold ─┬─► Postgres ──► Metabase
                                                                  └─► chunks ──► pgvector (próximo módulo)
```

**Regra da Bronze:** guardar como veio. Nada de limpar aqui. Só acrescentamos 3 colunas para saber *de onde* e *quando* veio:
`_ingest_ts`, `_source_file`, `_batch_id`.

**Antes de começar:** `docker compose up -d` (pasta `docker/`) e `pip install -r requirements.txt`.

In [2]:
import os
os.chdir(r"c:\Users\alexa\projeto_wuldson\arqu_coleta_dados_pos_unipe")
print("Pasta atual:", os.getcwd())

Pasta atual: c:\Users\alexa\projeto_wuldson\arqu_coleta_dados_pos_unipe


## Passo 0 · O ambiente está de pé?

In [3]:
import sys, subprocess

r = subprocess.run([sys.executable, "scripts/verifica_ambiente.py"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)

OK   Java 
OK   PySpark 3.5.3 
OK   MinIO (buckets landing e warehouse) 
OK   Spark 3.5.3 (grava e lÃª Parquet) 

Postgres + pgvector e Metabase: vamos montar juntos nas prÃ³ximas etapas.
Tudo pronto.
ÒXITO: o processo com PID 24172 (processo filho de PID 22648) foi finalizado.
ÒXITO: o processo com PID 22648 (processo filho de PID 1416) foi finalizado.
ÒXITO: o processo com PID 1416 (processo filho de PID 17956) foi finalizado.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

[Stage 0:>                                                          (0 + 0) / 2]

[Stage 0:>                                                          (0 + 2) / 2]

                                                                                

[Stage 2:>                                                          (0 + 2) / 2]

                                                                                



In [5]:
import os, sys
from pathlib import Path

# 1. Procura a raiz subindo até achar common/lake_local.py
def _achar_raiz():
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "common" / "lake_local.py").exists():
            return p
    # 2. Fallback: caminho fixo da sua máquina
    fixo = Path(r"c:\Users\alexa\projeto_wuldson\arqu_coleta_dados_pos_unipe")
    if (fixo / "common" / "lake_local.py").exists():
        return fixo
    return cur

ROOT = _achar_raiz()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Projeto:", ROOT)

from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F

Projeto: c:\Users\alexa\projeto_wuldson\arqu_coleta_dados_pos_unipe


## Passo 0.5 · Baixar as dependências do Spark (só na primeira vez)

O Spark precisa de alguns jars (Iceberg, S3A com o AWS SDK, JDBC). Na **primeira execução** ele os baixa da internet, e isso pode levar **vários minutos**, principalmente com a rede da sala dividida. Os arquivos ficam guardados em cache (pasta `.ivy2`) e, depois disso, o Spark abre rápido.

Rode a célula abaixo **em casa, antes da aula**. Ela abre um processo separado, baixa os jars e termina, sem deixar o Spark aberto neste notebook.

In [ ]:
import sys, subprocess

r = subprocess.run([sys.executable, "scripts/preparar_jars.py"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])  # fim da saída: deve terminar com "Jars baixados e prontos"
print(r.stderr[-1500:])  # se houver erro, a causa aparece aqui

## Passo 1 · Abrir o Spark (modo local)

O Spark roda **dentro deste Python**, com `local[2]` (2 núcleos) e 2 GB. Na primeira vez ele baixa os jars (Iceberg, S3A, JDBC). O S3A traz junto o AWS SDK, que é grande, então pode levar vários minutos e precisa de internet. Os jars ficam em cache em ~/.ivy2 e nas próximas execuções é rápido.
Para mais folga, `SPARK_CORES` e `SPARK_MEMORY` (variáveis de ambiente antes de abrir o Jupyter).
Spark UI: <http://localhost:4040> enquanto houver um job rodando.

In [2]:
import os
os.chdir(r"c:\Users\alexa\projeto_wuldson\arqu_coleta_dados_pos_unipe")
os.environ["MINIO_USER"] = "admin"
os.environ["MINIO_PASSWORD"] = "admin123"
print("Pasta atual:", os.getcwd())
print("MinIO user:", os.environ["MINIO_USER"])

Pasta atual: c:\Users\alexa\projeto_wuldson\arqu_coleta_dados_pos_unipe
MinIO user: admin


In [2]:
import os, sys
sys.path.insert(0, os.path.abspath("../.."))  # já estamos na raiz do projeto
from common import lake_local as L
from pyspark.sql import functions as F
spark = L.get_spark()
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

Spark 3.5.3 | master: local[2] | storage: local


## Passo 2 · Os arquivos chegam na landing (object storage)

In [3]:
n = L.upload_to_landing(L.DATA_DIR / "lote1", "mercadonordeste/lote1")
print(n, "arquivos enviados")
for chave, tam in L.list_landing("mercadonordeste/lote1"):
    print(f"{tam:>10,} bytes  {chave}")

4 arquivos enviados
   529,223 bytes  mercadonordeste\lote1\avaliacoes.csv
   420,891 bytes  mercadonordeste\lote1\clientes.csv
 7,601,232 bytes  mercadonordeste\lote1\pedidos.jsonl
     9,044 bytes  mercadonordeste\lote1\produtos.csv


Abra o console do MinIO (<http://localhost:9001>, com o usuário e a senha do seu `.env`) e veja o bucket **landing**.
É um *object storage*: não há pastas de verdade, só chaves com `/` no nome.
Você achou os objetos no bucket warehouse do MinIO, em lakehouse/bronze/<tabela>/data e metadata.

### Experimento: o que é uma "pasta" aqui?

Vamos pedir ao MinIO as chaves de três formas diferentes e fazer o experimento de **renomear**.
Antes de rodar, pense: se não existem pastas, como o console mostra `mercadonordeste` e `lote1` como pastas?

In [4]:
s3 = L._s3()     # cliente S3 (boto3) que o módulo usa para falar com o MinIO

# 1) "Pastas" são só prefixos: pedimos ao MinIO para agrupar as chaves pelo "/"
resp = s3.list_objects_v2(Bucket="landing", Prefix="mercadonordeste/", Delimiter="/")
print("Prefixos:", [p["Prefix"] for p in resp.get("CommonPrefixes", [])])

# 2) Listar por prefixo
for chave, tam in L.list_landing("mercadonordeste/lote1/"):
    print(f"{tam:>10,} bytes  {chave}")

# 3) Renomear não existe: é COPIAR e APAGAR (numa chave de teste, sem mexer nos dados)
s3.put_object(Bucket="landing", Key="teste/a.txt", Body=b"ola")
s3.copy_object(Bucket="landing", Key="teste/b.txt",
               CopySource={"Bucket": "landing", "Key": "teste/a.txt"})
s3.delete_object(Bucket="landing", Key="teste/a.txt")
print("Depois do 'renomear':", [c for c, _ in L.list_landing("teste/")])
s3.delete_object(Bucket="landing", Key="teste/b.txt")      # limpa o teste

#a “pasta” é só um prefixo, e o MinIO a simula quando pedimos o agrupamento por /. 
# Renomear são duas operações (copiar e apagar). 
# Em um bucket com milhões de objetos, “renomear uma pasta” significa copiar tudo, e isso vai pesar 
# mais adiante, quando falarmos de tabelas.

Prefixos: ['mercadonordeste/lote1/']
   529,223 bytes  mercadonordeste\lote1\avaliacoes.csv
   420,891 bytes  mercadonordeste\lote1\clientes.csv
 7,601,232 bytes  mercadonordeste\lote1\pedidos.jsonl
     9,044 bytes  mercadonordeste\lote1\produtos.csv
Depois do 'renomear': []


{'ResponseMetadata': {'RequestId': '18E595656A39F9F0',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18E595656A39F9F0',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '1806',
   'x-ratelimit-remaining': '1806',
   'x-xss-protection': '1; mode=block',
   'date': 'Fri, 06 Nov 2026 22:31:29 GMT'},
  'RetryAttempts': 0}}

## Passo 3 · Landing → Bronze

In [5]:
from common import medalhao as M

contagens = M.ingest_lote(spark, "lote1", L.landing_uri)
contagens

{'clientes': 5000, 'produtos': 200, 'pedidos': 30150, 'avaliacoes': 6489}

In [6]:
spark.sql("SHOW TABLES IN lake.bronze").show()
spark.table("lake.bronze.pedidos").printSchema()
spark.table("lake.bronze.pedidos").select("pedido_id","cliente_id","status","itens","_batch_id","_source_file").show(3, truncate=60)

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|   bronze|  clientes|      false|
|   bronze|avaliacoes|      false|
|   bronze|  produtos|      false|
|   bronze|   pedidos|      false|
+---------+----------+-----------+

root
 |-- cliente_id: long (nullable = true)
 |-- data_pedido: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- itens: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- preco_unit: double (nullable = true)
 |    |    |-- produto_id: long (nullable = true)
 |    |    |-- qtd: long (nullable = true)
 |-- pedido_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)

+---------+----------+---------+--------------------------------------------------+---------+-----------

### Onde está a tabela `pedidos`?

No Spark ela é `lake.bronze.pedidos`. No storage, é um conjunto de **objetos**. Vamos olhar:
`data/` guarda os arquivos Parquet (as linhas) e `metadata/` guarda o que o Iceberg usa para saber quais arquivos formam a tabela em cada versão.
Repare que `_batch_id=lote1` aparece **dentro da chave**: a tabela foi particionada por lote.

In [8]:
def resumo_objetos(prefixo):
    objs = L.list_warehouse(prefixo)
    return {"objetos": len(objs),
            "data (parquet)": sum("/data/" in c for c, _ in objs),
            "metadata": sum("/metadata/" in c for c, _ in objs),
            "KB": round(sum(t for _, t in objs) / 1024)}

for chave, tam in L.list_warehouse("lakehouse/bronze/pedidos")[:10]:
    print(f"{tam:>9,}  {chave}")
print(resumo_objetos("lakehouse/bronze/pedidos"))

# não existe “o arquivo da tabela”. 
# Uma tabela é o catálogo mais os arquivos de dados e de metadados. 
# Quem consulta pelo Spark vê uma tabela, e o storage só enxerga objetos.

{'objetos': 0, 'data (parquet)': 0, 'metadata': 0, 'KB': 0}


Repare: `itens` chegou como **lista de structs** (o JSON foi preservado) e os CSVs chegaram como texto. A tipagem é problema da Silver.

## Passo 4 · Idempotência
Rode o Passo 3 de novo. As contagens **não** dobram: a ingestão apaga o lote e regrava. Pipeline que duplica ao reexecutar é pipeline frágil.

In [9]:
prefixo = "lakehouse/bronze/pedidos"

linhas_antes  = spark.table("lake.bronze.pedidos").count()
objetos_antes = resumo_objetos(prefixo)

M.ingest_lote(spark, "lote1", L.landing_uri)          # reexecuta o MESMO lote

linhas_depois  = spark.table("lake.bronze.pedidos").count()
objetos_depois = resumo_objetos(prefixo)

print("Linhas  :", linhas_antes, "->", linhas_depois, "| idempotente:", linhas_antes == linhas_depois)
print("Objetos :", objetos_antes["objetos"], "->", objetos_depois["objetos"])
print("  data     :", objetos_antes["data (parquet)"], "->", objetos_depois["data (parquet)"])
print("  metadata :", objetos_antes["metadata"], "->", objetos_depois["metadata"])
spark.sql("SELECT count(*) AS arquivos_na_versao_atual FROM lake.bronze.pedidos.files").show()

Linhas  : 30150 -> 30150 | idempotente: True
Objetos : 0 -> 0
  data     : 0 -> 0
  metadata : 0 -> 0
+------------------------+
|arquivos_na_versao_atual|
+------------------------+
|                       1|
+------------------------+



### O que aconteceu?

As **linhas** não dobraram (idempotência), mas os **objetos** no storage aumentaram. Por quê?
O storage não edita arquivos: ele só cria objetos novos. O Iceberg registra uma nova versão da tabela e deixa de *apontar* para os arquivos antigos, mas eles continuam lá, guardando o passado. A última linha mostra quantos arquivos fazem parte da versão **atual**, comparando com o total de objetos que o storage guarda.
Isso é o que permite voltar no tempo e, ao mesmo tempo, o motivo de existir limpeza de arquivos antigos.

## Checkpoint
- [ ] `lake.bronze` tem `clientes`, `produtos`, `pedidos`, `avaliacoes`.
- [ ] Reexecutar não duplica.
- [ ] Você achou `data/` e `metadata/` em `lakehouse/bronze/<tabela>` e explica por que os objetos aumentaram ao reexecutar.

➡️ Próximo: `02_silver.ipynb`